# Membuat Edges (Hubungan Guru dan Murid)

## Import Library

In [5]:
import pandas as pd
import ast

## Load Data

In [6]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)
df.columns = ['Hadith', 'Book', 'Num_hadith', 'Matn', 'Sanad', 'Sanad_Length', 'Sanad_no_harakat']
df = df.iloc[1:].reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16200\1150346939.py:3: DtypeWarning: Columns (0: 2, 1: 5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)


### Menghitung data dan Memecah isi list pada kolom Sanad_no_harakat

In [7]:
df['Sanad_no_harakat'] = df['Sanad_no_harakat'].apply(ast.literal_eval)


In [8]:
df.shape

(487451, 7)

In [9]:
print(type(df['Sanad_no_harakat'].iloc[0]))

<class 'list'>


In [10]:
sanad_exploded = df.explode('Sanad_no_harakat')


In [11]:
sanad_exploded.shape

(2993324, 7)

In [14]:
df.Sanad_no_harakat.head(10)

0    [ابو عبيدة مسلم بن ابو كريمة التميمي, جابر بن ...
1                      [ابو عبيدة, جابر بن زيد, عائشة]
2                             [ابو عبيدة, جابر بن زيد]
3                  [ابو عبيدة, جابر بن زيد, ابو هريرة]
4            [ابو عبيدة, جابر بن زيد, ابو سعيد الخدري]
5                   [ابو عبيدة, جابر بن زيد, ابن عباس]
6                [ابو عبيدة, جابر بن زيد, انس بن مالك]
7            [ابو عبيدة, جابر بن زيد, ابو سعيد الخدري]
8                  [ابو عبيدة, جابر بن زيد, ابو هريرة]
9              [ابو عبيدة, جابر بن زيد, عمر بن الخطاب]
Name: Sanad_no_harakat, dtype: object

### Menghitung Perawi (Narrator) Paling Banyak Muncul

In [15]:
# Hitung frekuensi setiap perawi
from collections import Counter

# Flatten semua nama perawi dari list Sanad_no_harakat
all_narrators = []
for sanad_list in df['Sanad_no_harakat']:
    all_narrators.extend(sanad_list)

# Hitung frekuensi
narrator_counts = Counter(all_narrators)

# Buat DataFrame dari hasil perhitungan
narrator_df = pd.DataFrame(narrator_counts.most_common(), columns=['Perawi', 'Jumlah'])
narrator_df

,Perawi,Jumlah
0,ابو هريرة,51623
1,ابن عباس,47925
2,عائشة,33019
3,ابن عمر,28671
4,شعبة,26324
...,...,...
191599,الاشعث بن طلق,1
191600,وحفص بن موسي,1
191601,الحجاج <IDF> يعني<IDF> الصواف,1
191602,وابي مسعود البدري,1


### Buat Hubungan Guru-Murid dari Sanad


In [22]:
def build_teacher_student_edges(chain):
    if not isinstance(chain, list) or len(chain) < 2:
        return []
    # Asumsi: daftar sanad ditulis dari perawi awal ke perawi akhir,
    # sehingga guru adalah elemen berikutnya dalam rantai.
    return [(chain[i], chain[i + 1]) for i in range(len(chain) - 1)]

# Pastikan semua nilai Sanad_no_harakat sudah berupa list
# Jika ada baris yang masih berupa string, ubah menjadi list terlebih dahulu

def ensure_list(cell):
    if isinstance(cell, list):
        return cell
    if isinstance(cell, str) and cell.strip().startswith('['):
        try:
            return ast.literal_eval(cell)
        except Exception:
            return [cell]
    if pd.isna(cell):
        return []
    return [cell]

sanad_lists = df['Sanad_no_harakat'].apply(ensure_list)

edges = []
for sanad in sanad_lists:
    edges.extend(build_teacher_student_edges(sanad))

edges_df = pd.DataFrame(edges, columns=['Guru', 'Murid'])

# Tambahkan frekuensi kemunculan hubungan guru-murid
edges_freq = (
    edges_df
    .value_counts()
    .reset_index(name='Jumlah')
    .sort_values(by='Jumlah', ascending=False)
)

print("Contoh hubungan guru-murid:")
print(edges_df.head(15))
print("\nHubungan guru-murid paling sering muncul:")
print(edges_freq.head(20))

print(f"\nTotal edge guru-murid: {len(edges_df)}")
print(f"Total hubungan unik: {len(edges_freq)}")


Contoh hubungan guru-murid:
                                   Guru               Murid
0   ابو عبيدة مسلم بن ابو كريمة التميمي  جابر بن زيد الازدي
1                    جابر بن زيد الازدي    عبد الله بن عباس
2                             ابو عبيدة         جابر بن زيد
3                           جابر بن زيد               عائشة
4                             ابو عبيدة         جابر بن زيد
5                             ابو عبيدة         جابر بن زيد
6                           جابر بن زيد           ابو هريرة
7                             ابو عبيدة         جابر بن زيد
8                           جابر بن زيد     ابو سعيد الخدري
9                             ابو عبيدة         جابر بن زيد
10                          جابر بن زيد            ابن عباس
11                            ابو عبيدة         جابر بن زيد
12                          جابر بن زيد         انس بن مالك
13                            ابو عبيدة         جابر بن زيد
14                          جابر بن زيد     ابو سعيد الخدري

Hubungan gu